可以。你现在已经理解 MOTR 的主线了，下一步确实应该把 **MOT 领域的“基础词汇体系”补齐**。否则后面看 ByteTrack、DeepSORT、TrackFormer、MOTRv2、Sparse4D 等论文时，很容易出现“每个单词认识，但整句话不知道在干什么”的情况。

我建议把这些概念放到一条完整流水线里理解：

$$
\boxed{
\text{Detection}
\rightarrow
\text{Motion Prediction}
\rightarrow
\text{Appearance/ReID}
\rightarrow
\text{Similarity}
\rightarrow
\text{Data Association}
\rightarrow
\text{Track Management}
}
$$

传统 MOT 的核心任务就是：**在每一帧找到目标，并且保证同一个真实目标在不同帧里始终使用同一个 ID。** ([科学直通车][1])

---

# 1. MOT 是什么？

## MOT = Multiple Object Tracking

中文：

> **多目标跟踪**

有时候也叫：

$$
MTT = Multiple\ Target\ Tracking
$$

给定连续视频：

$$
I_1,I_2,\dots,I_T
$$

MOT 不仅需要检测：

```text
Frame 1

┌───────┐
│ Car   │
└───────┘

┌───────┐
│ Car   │
└───────┘
```

还必须告诉你：

```text
Frame 1          Frame 2          Frame 3

Car ID=1   →     Car ID=1   →     Car ID=1

Car ID=2   →     Car ID=2   →     Car ID=2
```

所以 MOT 可以概括为：

$$
\boxed{
MOT = Detection + Identity Association
}
$$

或者更准确：

$$
\boxed{
MOT =
Localization
+
Association
+
Track\ Management
}
$$

MOT 的目标是找到多个目标、维持它们的 identity，并最终形成每个目标随时间变化的轨迹。([科学直通车][1])

---

# 2. Detection 和 Tracking 到底有什么区别？

这是最基础的一组概念。

## Object Detection

目标检测解决：

> **这一张图片里面有什么？在哪里？**

例如 YOLO 输入：

$$
I_t
$$

输出：

$$
D_t=
\{d_1,d_2,\dots,d_N\}
$$

其中：

$$
d_i=(x,y,w,h,class,score)
$$

例如：

```text
Frame 10

Detection 1:
bbox = [100,200,50,100]
class = person
score = 0.95

Detection 2:
bbox = [400,220,55,105]
class = person
score = 0.91
```

但是 Detection 本身不知道：

> Detection 1 是不是 Frame 9 里面的那个人？

---

## Tracking

Tracking 多一个：

$$
\boxed{ID}
$$

输出变成：

$$
(x,y,w,h,class,score,\color{#555}{ID})
$$

例如：

```text
Frame 10:

Person
bbox = [...]
ID = 17
```

下一帧：

```text
Frame 11:

Person
bbox = [...]
ID = 17
```

即使位置发生变化：

$$
(x_t,y_t)\neq(x_{t+1},y_{t+1})
$$

仍然应该：

$$
ID_t=ID_{t+1}
$$

所以 tracking 最关键的问题其实不是“框出来”，而是：

$$
\boxed{\text{Who is who?}}
$$

---

# 3. SOT 和 MOT

还经常看到一个：

$$
SOT = Single\ Object\ Tracking
$$

即：

> **单目标跟踪**

区别非常简单。

SOT 通常是：

```text
第一帧：

这是我要追踪的人
        ↓
     bbox
        ↓
后面的帧一直找他
```

所以目标通常已经指定。

而 MOT：

```text
Frame t

Person A
Person B
Person C
Person D
...
```

目标数量：

$$
N_t
$$

本身就是动态变化的。

所以 MOT 比 SOT 多了非常核心的问题：

$$
\boxed{Data\ Association}
$$

以及目标的 birth/death、遮挡、ID 维护等。([科学直通车][1])

---

# 4. Tracking-by-Detection 是什么？

这是理解 MOTR 前最重要的传统范式。

简称：

$$
\boxed{TBD}
$$

即：

> **Tracking by Detection**

核心思想：

> 每帧先检测，再把不同帧的 detection 关联起来。

流程：

```text
              Frame t
                 ↓
              Detector
                 ↓
            Detections
                 ↓
      ┌──────────┴──────────┐
      ↓                     ↓
 Motion Prediction      Appearance
 Kalman Filter             ReID
      ↓                     ↓
      └──────────┬──────────┘
                 ↓
          Similarity / Cost
                 ↓
             Matching
                 ↓
        Hungarian Algorithm
                 ↓
             Track IDs
```

典型代表包括：

* SORT
* DeepSORT
* ByteTrack

而 MOTR 的核心创新之一，就是想摆脱这套显式 association pipeline。

---

# 5. Track 是什么？

这个词后面会出现无数次。

一个：

$$
\boxed{Track}
$$

就是某一个目标随时间变化的状态。

例如：

```text
Car ID = 7

Frame 10: (100,200)
Frame 11: (105,201)
Frame 12: (111,202)
Frame 13: (118,203)
```

可以写成：

$$
T_7=
\{
b_7^{10},
b_7^{11},
b_7^{12},
b_7^{13}
\}
$$

这里 \(b\) 是 bounding box。

实际 tracker 内部的 Track 往往还会保存：

```text
Track {
    ID
    bbox
    velocity
    age
    confidence
    appearance_feature
    lost_frames
    ...
}
```

所以 Track 不只是一个 bbox。

它更像：

$$
\boxed{\text{某个目标在 Tracker 中的状态}}
$$

这和你做车辆控制时的 VehicleState 概念很像。

---

# 6. Tracklet 又是什么？

这个词也特别常见。

$$
\boxed{Tracklet = Short\ Track}
$$

即：

> 一小段连续的目标轨迹。

例如真实轨迹：

```text
Frame:

1  2  3  4  5  6  7  8  9  10

A--A--A--A-----遮挡-----A--A--A
```

tracker 可能得到：

```text
Tracklet 1:

A--A--A--A


Tracklet 2:

                 A--A--A
```

那么问题就是：

$$
Tracklet_1
\stackrel{?}{=}
Tracklet_2
$$

如果判断是同一个人：

$$
Tracklet_1 + Tracklet_2
\rightarrow
Track
$$

这就是：

$$
\boxed{Tracklet Association}
$$

---

# 7. Data Association 是什么？

这是整个 MOT 最核心的概念之一。

全称：

$$
\boxed{Data\ Association}
$$

中文通常叫：

> **数据关联**

意思：

> 当前帧 detection 和历史 track 应该怎么对应？

假设上一帧：

```text
Track 1
Track 2
Track 3
```

这一帧 detector 得到：

```text
Detection A
Detection B
Detection C
```

我们需要判断：

```text
Track 1 ↔ Detection B
Track 2 ↔ Detection C
Track 3 ↔ Detection A
```

数学上：

$$
T=\{T_1,T_2,T_3\}
$$

$$
D=\{D_A,D_B,D_C\}
$$

需要求一个 assignment：

$$
\pi:T\rightarrow D
$$

这就是：

$$
\boxed{Association}
$$

MOT 综述通常把它描述为两个问题：先定义不同观测之间的相似性，再根据这些相似性恢复跨帧 identity。([科学直通车][1])

---

# 8. 那怎么判断两个目标是不是同一个？

这就引出你刚才提到的：

$$
\boxed{Similarity}
$$

一般主要有两类：

$$
\boxed{
Appearance\ Similarity
+
Motion\ Similarity
}
$$

也就是：

```text
这个人“长得像不像”？
        +
这个人的“运动对不对”？
```

这两个是传统 MOT 最重要的 association cues。([科学直通车][1])

---

# 9. Appearance 是什么？

Appearance：

> **外观**

指一个目标视觉上“长什么样”。

比如：

```text
Person A

黑色衣服
白色鞋
蓝色裤子
背包
身材
纹理
颜色
局部特征
...
```

但神经网络当然不会真的存：

```text
"黑衣服 + 白鞋"
```

而是提取：

$$
f_{app}\in\mathbb R^d
$$

例如：

$$
d=128
$$

那么：

$$
f=
[0.12,-0.43,0.81,\dots,0.27]
$$

这个向量叫：

$$
\boxed{Appearance\ Feature}
$$

或者：

$$
\boxed{Appearance\ Embedding}
$$

综述通常把 appearance model 分为两部分：**visual representation** 和衡量两个 representation 是否相似的 **statistical measurement**。([科学直通车][1])

---

# 10. Appearance Similarity

有了两个 feature：

$$
f_i,\quad f_j
$$

就可以比较：

> 长得像不像？

最常见的是：

$$
\boxed{Cosine\ Similarity}
$$

公式：

$$
sim(f_i,f_j)
=
\frac{f_i^Tf_j}
{\|f_i\|\|f_j\|}
$$

如果两个 embedding 很接近：

$$
sim\rightarrow1
$$

那么：

$$
\boxed{\text{可能是同一个目标}}
$$

例如：

$$
sim(A,B)=0.95
$$

很像。

而：

$$
sim(A,C)=0.15
$$

很不像。

因此：

$$
\boxed{
Appearance\ Similarity
=
\text{视觉特征相似程度}
}
$$

---

# 11. ReID 是什么？

这是非常重要的。

全称：

$$
\boxed{ReID = Re-Identification}
$$

完整一些：

$$
\boxed{Person\ Re-Identification}
$$

即：

> **行人重识别**

如果对象是车辆，则叫：

$$
Vehicle\ ReID
$$

---

## ReID 在解决什么问题？

假设：

```text
Camera / Frame

Person A
   ↓
   ↓
被汽车遮挡
   ↓
   ↓
重新出现
```

问题：

> 重新出现的这个人，是不是之前的 Person A？

ReID 就干这个。

---

# 12. ReID 网络在做什么？

输入一个目标 crop：

```text
┌─────────┐
│         │
│ Person  │
│         │
└─────────┘
```

经过 CNN / Transformer：

$$
image
\rightarrow
ReID\ Network
\rightarrow
Embedding
$$

例如：

$$
f_A\in\mathbb R^{128}
$$

另一个目标：

$$
f_B\in\mathbb R^{128}
$$

计算：

$$
cos(f_A,f_B)
$$

如果：

$$
cos(f_A,f_B)=0.96
$$

说明很像。

ReID 的目标就是学习一个 embedding space，使：

$$
\boxed{
\text{Same ID}
\Rightarrow
\text{small distance}
}
$$

而：

$$
\boxed{
\text{Different ID}
\Rightarrow
\text{large distance}
}
$$

深度 ReID 常被用于遮挡后的 track recovery 和较长时间间隔的身份匹配。([数字对象标识符][2])

---

# 13. 为什么 Motion 不够，还需要 ReID？

假设：

```text
          A →
          B →
```

两个行人距离非常近。

下一帧：

```text
           ?
           ?
```

如果只根据位置：

$$
distance(A,D_1)
$$

和：

$$
distance(B,D_1)
$$

可能差不多。

但是：

```text
A = 白衣服
B = 黑衣服
```

ReID：

$$
sim(A,D_1)=0.95
$$

$$
sim(B,D_1)=0.12
$$

于是很好判断：

$$
D_1=A
$$

所以：

$$
\boxed{
Motion = Where should it be?
}
$$

$$
\boxed{
Appearance = What does it look like?
}
$$

两者互补。

---

# 14. Motion Model 是什么？

Motion：

> **运动信息**

比如一个车：

```text
t-2      t-1       t

x=10 → x=15 → ?
```

如果速度基本恒定：

$$
v=5
$$

那么预测：

$$
x_t\approx20
$$

这就是：

$$
\boxed{Motion\ Prediction}
$$

MOT 中经常假设短时间内目标运动较平滑，用运动模型预测下一帧目标可能出现的位置，从而缩小 association 的搜索范围。([科学直通车][1])

---

# 15. Kalman Filter 为什么经常出现？

经典 MOT 经常使用：

$$
\boxed{Kalman\ Filter}
$$

中文：

> 卡尔曼滤波器

比如 SORT 就大量依赖它。

可以维护一个状态：

$$
x_t=
[x,y,w,h,v_x,v_y,\dots]^T
$$

然后根据：

$$
x_t=Fx_{t-1}+w
$$

预测：

$$
\hat x_t
$$

例如：

```text
Frame t-2      Frame t-1      Frame t

  □  →            □  →         ?
```

Kalman Filter 预测：

```text
                            □
                         predicted
```

然后 detector 真正检测到：

```text
                             □
                          detection
```

两者很近：

$$
\boxed{\text{很可能是同一个目标}}
$$

---

# 16. Motion Similarity 是什么？

假设 Track A 预测 bbox：

$$
B_A^{pred}
$$

当前有 Detection：

$$
B_D
$$

可以计算：

$$
IoU(B_A^{pred},B_D)
$$

如果：

$$
IoU=0.9
$$

说明位置高度一致。

所以：

$$
\boxed{Motion\ Similarity}
$$

可以理解为：

> 当前 detection 是否符合这个 Track 的运动预测？

常见方式包括：

$$
IoU
$$

$$
Euclidean\ Distance
$$

$$
Mahalanobis\ Distance
$$

等等。([预印本平台][3])

---

# 17. IoU 是什么？

全称：

$$
\boxed{Intersection\ over\ Union}
$$

中文：

> 交并比

两个 bbox：

$$
A,\quad B
$$

定义：

$$
IoU(A,B)
=
\frac{|A\cap B|}
{|A\cup B|}
$$

如果完全重合：

$$
IoU=1
$$

完全不重合：

$$
IoU=0
$$

例如：

```text
┌──────────┐
│ A        │
│    ┌──────────┐
│    │ overlap  │
└────│          │
     │ B        │
     └──────────┘
```

重叠越大：

$$
IoU\uparrow
$$

说明：

$$
\boxed{\text{越可能是同一个目标}}
$$

在传统 tracking 中经常：

$$
Track_{pred}
\leftrightarrow
Detection
$$

计算 IoU。

---

# 18. Cost Matrix / Affinity Matrix

这也是论文里高频词。

假设：

$$
3\ Tracks
$$

和：

$$
3\ Detections
$$

可以构造：

$$
C=
\begin{bmatrix}
C_{11}&C_{12}&C_{13}\\
C_{21}&C_{22}&C_{23}\\
C_{31}&C_{32}&C_{33}
\end{bmatrix}
$$

例如：

$$
C=
\begin{bmatrix}
0.1&0.9&0.8\\
0.7&0.2&0.9\\
0.8&0.7&0.1
\end{bmatrix}
$$

其中：

$$
C_{ij}
$$

表示：

$$
Track_i
\leftrightarrow
Detection_j
$$

的匹配代价。

越小越好。

那么自然得到：

$$
T_1\leftrightarrow D_1
$$

$$
T_2\leftrightarrow D_2
$$

$$
T_3\leftrightarrow D_3
$$

---

# 19. Appearance + Motion 怎么融合？

例如定义：

$$
C_{ij}
=
\lambda C_{appearance}
+
(1-\lambda)C_{motion}
$$

其中：

$$
C_{appearance}
=
1-\cos(f_i,f_j)
$$

而：

$$
C_{motion}
=
1-IoU(B_i,B_j)
$$

所以：

$$
\boxed{
C_{ij}
=
\lambda
(1-\cos(f_i,f_j))
+
(1-\lambda)
(1-IoU(B_i,B_j))
}
$$

直观理解：

```text
             Track A
                │
       ┌────────┴────────┐
       ↓                 ↓
  长得像不像？       位置对不对？
 Appearance           Motion
       │                 │
       └────────┬────────┘
                ↓
              Cost
                ↓
        Detection B
```

实际算法会比这个复杂，但基本思想就是这样。

---

# 20. Hungarian Algorithm 是什么？

全称：

$$
\boxed{Hungarian\ Algorithm}
$$

中文：

> **匈牙利算法**

它解决的是 assignment problem：

> 给定 Tracks 和 Detections 的 cost matrix，怎么做全局的一对一匹配，使总 cost 最小？

例如：

$$
C=
\begin{bmatrix}
1&8&9\\
7&2&8\\
9&6&1
\end{bmatrix}
$$

最佳方案：

$$
T_1\rightarrow D_1
$$

$$
T_2\rightarrow D_2
$$

$$
T_3\rightarrow D_3
$$

总 cost：

$$
1+2+1=4
$$

所以传统 MOT 经常是：

$$
\boxed{
Similarity
\rightarrow
Cost\ Matrix
\rightarrow
Hungarian
\rightarrow
Association
}
$$

Hungarian 是 MOT 中非常常见的全局 assignment 方法。([预印本平台][3])

---

# 21. NMS 是什么？

全称：

$$
\boxed{NMS = Non-Maximum Suppression}
$$

中文：

> **非极大值抑制**

它原本主要用于目标检测。

例如 detector 对同一个人输出：

```text
bbox A: score=0.95
bbox B: score=0.91
bbox C: score=0.84
```

三个框其实都是一个人。

如果：

$$
IoU(A,B)>threshold
$$

而 A 分数最高：

$$
score_A>score_B
$$

那么：

```text
保留 A
删除 B
```

于是：

$$
\boxed{
NMS =
删除同一个目标的重复检测框
}
$$

---

# 22. Track NMS 又是什么？

Track NMS 可以理解成：

> **在 Track 层面做重复抑制。**

假设 tracker 错误地产生：

```text
Track ID=12 → Person A

Track ID=35 → Person A
```

实际上：

$$
ID12=ID35
$$

但系统产生了两个 Track。

那么可以根据：

* bbox overlap
* track confidence
* trajectory overlap
* temporal overlap

等信息判断：

$$
Track_{12}
$$

和：

$$
Track_{35}
$$

是不是重复。

如果重复：

```text
保留高质量 Track
删除另一个
```

这就是 Track NMS 的基本思想。

所以：

$$
NMS
$$

主要处理：

$$
\boxed{Duplicate\ Detection}
$$

而：

$$
Track\ NMS
$$

处理：

$$
\boxed{Duplicate\ Track}
$$

---

# 23. 为什么 MOTR 强调“不需要 Track NMS”？

因为 MOTR 希望：

$$
\boxed{One\ Query \leftrightarrow One\ Object}
$$

通过 Transformer 的 set prediction 和 TALA，让不同 Query 学会竞争不同目标。

理想情况下：

```text
q1 → Person A
q2 → Person B
q3 → Person C
```

而不是：

```text
q1 ─┐
    ├→ Person A
q2 ─┘
```

所以它希望：

$$
\boxed{\text{从模型结构/训练中避免 duplicate track}}
$$

而不是：

```text
先产生重复
↓
Track NMS
↓
后处理删掉
```

这就是所谓：

$$
\boxed{End-to-End}
$$

思想的一部分。

---

# 24. ID Switch / IDS 是什么？

这也是 MOT 最重要的错误类型之一。

全称：

$$
\boxed{ID\ Switch}
$$

简称：

$$
\boxed{IDS}
$$

例如真实 Person A：

```text
Frame 1    Frame 2    Frame 3    Frame 4

 ID=7  →    ID=7  →    ID=7  →    ID=21
```

那么 Frame 4：

$$
7\rightarrow21
$$

发生：

$$
\boxed{ID\ Switch}
$$

另一种典型情况：

```text
A = ID 1
B = ID 2

两个人交叉

A = ID 2
B = ID 1
```

也发生了 identity switch。

MOT 中遮挡、目标外观相似等都容易造成 ID switch。([MDPI][4])

---

# 25. Occlusion 是什么？

全称：

$$
\boxed{Occlusion}
$$

中文：

> **遮挡**

例如：

```text
Person A
    ↓
    ↓
┌───────────┐
│    BUS    │
└───────────┘
    ↓
    ↓
Person A
```

中间几帧 Person A 看不见。

问题：

> 再出现的时候，怎么知道还是 Person A？

这时候：

$$
Motion
$$

可能已经不可靠。

而：

$$
Appearance/ReID
$$

就很重要。

因此 occlusion 是 MOT 的核心难题之一。([科学直通车][1])

---

# 26. Lost Track

目标消失几帧以后，通常不会立刻删除 Track。

例如：

```text
Frame 10: ID7 visible
Frame 11: ID7 visible
Frame 12: occluded
Frame 13: occluded
Frame 14: visible
```

Tracker 可以维护：

```text
Track 7

state = LOST
lost_frames = 2
```

而不是：

```text
delete Track 7
```

如果 Frame 14 匹配回来：

$$
Lost\rightarrow Active
$$

继续：

$$
ID=7
$$

---

# 27. Track Birth / Death

这个概念在 MOTR 的 QIM 里面特别重要。

### Birth

新目标进入：

```text
Frame 1:

A B

Frame 2:

A B C
    ↑
  new
```

那么：

$$
C
$$

需要创建新 Track：

$$
\boxed{Track\ Birth}
$$

MOTR 里：

$$
Detect\ Query
\rightarrow
Track\ Query
$$

本质上就是 Track Birth。

### Death

目标离开：

```text
A B C
↓
A   C
```

B 长时间不出现：

$$
\boxed{Track\ Death}
$$

删除：

$$
Track_B
$$

MOTR 的 QIM 也在负责这个问题。

---

# 28. False Positive / False Negative

也必须认识。

## FP = False Positive

假阳性。

模型说：

> 这里有人。

实际上：

> 没有人。

即：

$$
Prediction=Object
$$

但：

$$
GT=Background
$$

---

## FN = False Negative

假阴性。

实际上有人：

$$
GT=Object
$$

但 detector：

$$
Prediction=Nothing
$$

也就是：

> **漏检。**

对于 tracking，FN 特别麻烦：

```text
Frame 1: A
Frame 2: A
Frame 3: ×  ← FN
Frame 4: A
```

可能造成：

$$
Track\ Fragmentation
$$

甚至：

$$
ID\ Switch
$$

---

# 29. Track Fragmentation 是什么？

简称经常写：

$$
\boxed{Frag}
$$

假设真实轨迹：

```text
A──────────────A
```

Tracker：

```text
Track 1:
A────A

      missing

             Track 2:
             A────A
```

原本一个完整轨迹：

$$
T
$$

被切成：

$$
T_1,T_2
$$

这就是：

$$
\boxed{Track\ Fragmentation}
$$

---

# 30. Gating 是什么？

这个词看 Kalman Filter + Hungarian 时会经常遇到。

假设：

```text
Track A 在左边
```

而一个 detection：

```text
D 在 500 米外
```

那显然：

$$
A\not\leftrightarrow D
$$

根本没必要拿去 Hungarian Matching。

因此先设置一个：

$$
\boxed{Gate}
$$

例如：

$$
distance<d_{max}
$$

或者：

$$
IoU>IoU_{min}
$$

只有满足条件的候选：

$$
Track_i\leftrightarrow Detection_j
$$

才进入 association。

这叫：

$$
\boxed{Gating}
$$

即：

> **先排除明显不可能的匹配。**

---

# 31. Mahalanobis Distance 是什么？

这个在 DeepSORT 等方法里很常见。

普通欧氏距离：

$$
d(x,y)=\|x-y\|_2
$$

没有考虑预测的不确定性。

Mahalanobis Distance：

$$
d_M(x)
=
\sqrt{
(x-\mu)^T
\Sigma^{-1}
(x-\mu)
}
$$

其中：

$$
\mu
$$

是 Kalman Filter 预测的位置，

$$
\Sigma
$$

是预测 uncertainty。

因此它实际上问：

> 这个 detection 相对于我的预测分布，到底有多“不正常”？

比单纯：

$$
|x_{det}-x_{pred}|
$$

更加合理。

---

# 32. Online Tracking / Offline Tracking

## Online Tracking

只能使用：

$$
I_1,\dots,I_t
$$

预测当前结果。

不能看未来：

$$
I_{t+1}
$$

所以：

$$
\boxed{Causal}
$$

适合：

* 自动驾驶
* 机器人
* 实时监控

---

## Offline Tracking

可以：

$$
I_1,\dots,I_T
$$

整段视频一起看。

所以 Frame 10 的 tracking 可以利用：

```text
Frame 11
Frame 12
Frame 13
...
```

的信息。

通常可以做：

$$
Global\ Association
$$

但无法实时。

---

# 33. MOT 中所谓的 Appearance Model 和 Motion Model

现在可以把两个概念彻底区分开。

### Appearance Model

回答：

$$
\boxed{\text{Who does it look like?}}
$$

输入：

$$
Image\ Crop
$$

输出：

$$
Appearance\ Embedding
$$

然后：

$$
cos(f_i,f_j)
$$

进行 ReID。

### Motion Model

回答：

$$
\boxed{\text{Where should it be?}}
$$

输入：

$$
x_{t-1}
$$

预测：

$$
\hat x_t
$$

例如：

$$
Kalman\ Filter
$$

然后比较：

$$
\hat x_t
\leftrightarrow Detection
$$

---

# 34. 一个完整的传统 MOT Pipeline

现在把前面的知识全部串起来。

假设使用：

$$
YOLO + Kalman + ReID + Hungarian
$$

那么：

```text
                     Frame t
                        │
                        ▼
                     YOLO
                        │
                        ▼
                   Detections
                        │
           ┌────────────┴────────────┐
           │                         │
           ▼                         ▼
       ReID Network            Kalman Filter
           │                         │
           ▼                         ▼
   Appearance Feature         Predicted BBox
           │                         │
           ▼                         ▼
Appearance Similarity       Motion Similarity
           │                         │
           └────────────┬────────────┘
                        │
                        ▼
                   Cost Matrix
                        │
                        ▼
               Gating + Hungarian
                        │
                        ▼
                Data Association
                        │
          ┌─────────────┼─────────────┐
          ▼             ▼             ▼
       Matched       Unmatched     Unmatched
       Track         Detection       Track
          │             │             │
          ▼             ▼             ▼
       Update        New Track     Lost Track
                        │             │
                        │          too long?
                        │             │
                        │             ▼
                        │         Delete Track
                        │
                        ▼
                  Track Birth
```

**这张图非常重要。**

因为它基本就是：

$$
\boxed{\text{传统 Tracking-by-Detection 的标准范式}}
$$

---

# 35. 现在再回头看 MOTR，就会突然非常清楚

传统方法：

```text
Detection
    ↓
Appearance ─────┐
                │
Motion ─────────┤
                ↓
          Similarity
                ↓
          Cost Matrix
                ↓
          Hungarian
                ↓
         Association
                ↓
             Track
```

MOTR：

```text
Frame t
   ↓
Transformer
   ↓
Track Query t
   ↓
   ↓ temporal propagation
   ↓
Track Query t+1
```

所以 MOTR 真正想干的是：

$$
\boxed{
Appearance\ Matching
+
Motion\ Matching
+
Data\ Association
}
$$

从一堆**显式手工设计模块**，变成：

$$
\boxed{
Learned\ Query\ Propagation
}
$$

传统方法问：

> **“当前这个 Detection 到底应该和历史哪个 Track 匹配？”**

MOTR 的思路则是：

> **“我不想每帧重新找你是谁。我维护一个属于你的 Track Query，然后让它在下一帧继续找你。”**

所以传统 MOT：

$$
\boxed{
Detection
\rightarrow
Matching
\rightarrow
Track
}
$$

MOTR：

$$
\boxed{
Track\ Query_t
+
Image_{t+1}
\rightarrow
Track\ Query_{t+1}
}
$$

这就是为什么理解 **ReID、appearance similarity、motion similarity、Hungarian、Kalman Filter、Track NMS、Data Association** 之后，你会发现 MOTR 论文的动机一下子清楚很多：它实际上就是在尝试把这些传统 tracking pipeline 中割裂的模块，尽可能统一到一个端到端的时序 Transformer 里。MOT 文献本身也长期把 appearance、motion、occlusion handling 和 association/inference 看作核心组成部分。([科学直通车][1])

如果接下来继续深入 MOTR，我建议下一步重点补 **Kalman Filter → Hungarian Algorithm → SORT → DeepSORT → ByteTrack** 这条线。把这条传统 MOT 技术路线搞明白，再回来对比 **DETR → TrackFormer → MOTR → MOTRv2**，你会非常清楚为什么 Track Query 是一次很大的范式变化。

[1]: https://www.sciencedirect.com/science/article/abs/pii/S0004370220301958?utm_source=chatgpt.com "Multiple object tracking: A literature review - ScienceDirect"
[2]: https://doi.org/10.1016/J.NEUCOM.2023.126558?utm_source=chatgpt.com "Multi-camera multi-object tracking: A review of current trends and future advances - ScienceDirect"
[3]: https://www.preprints.org/manuscript/202608.0132?utm_source=chatgpt.com "Towards End-to-End Multi-Object Visual Tracking: A Survey and Outlook[v1] | Preprints.org"
[4]: https://www.mdpi.com/2079-9292/10/19/2406?utm_source=chatgpt.com "Multiple Object Tracking in Deep Learning Approaches: A Survey"
